In [1]:
# Parameters
run_id = "28b2de94-0f48-40bf-92ca-85995e1ba8da"
artifacts_dir = "/home/adnoman/projects/aml_gan/AMLend2end/artifacts/runs/28b2de94-0f48-40bf-92ca-85995e1ba8da"
sample_size = None
epochs = None
threshold = None


# Create node embeddings feature groups.

Up until now we use feature engineering, feature store and model training to create node embedding. We will now materialise this as node embeddings feature group. This feature group will be used to train anomaly detection model.

![Feature Stores](./images/online_offline_fs.png)

---
**NOTE**: 

In real life scenarios financial transaction are dynamically evolving graphs. If live Transaction Monitoring System is based on graph or node embeddings then this will require 1st to update the graph and node representations after new transactions arrive. Recomputing entire graph for every newly arrived transaction will lead to unaxeptable delayes and even monitoring system failures. This problem  will be more sever if large amount of updates happen in a short time window.

Contact us at Logical Clocks and we will help you to setup end to end graph based deep anomaly detection live Transaction Monitoring Systems. 

---

## Query Model Repository for best node embeddings model

In [2]:
# Setup for local execution
import os
import json
import pandas as pd
import numpy as np

# Define paths
BASE_PATH = os.path.dirname(os.path.abspath("__file__"))
TRAINING_DATA_PATH = os.path.join(BASE_PATH, "training_data")
OUTPUT_PATH = os.path.join(BASE_PATH, "output")
MODELS_PATH = os.path.join(BASE_PATH, "models")
RESOURCES_PATH = os.path.join(BASE_PATH, "Resources")

print(f"Training data: {TRAINING_DATA_PATH}")
print(f"Output: {OUTPUT_PATH}")

Training data: /home/adnoman/projects/aml_gan/AMLend2end/training_data
Output: /home/adnoman/projects/aml_gan/AMLend2end/output


In [3]:
# Find the latest model directory
model_dirs = [d for d in os.listdir(MODELS_PATH) if d.startswith('node_embeddings_')]
if model_dirs:
    latest_model_dir = os.path.join(MODELS_PATH, sorted(model_dirs)[-1])
    print(f"Found model: {latest_model_dir}")
    
    # Load metadata
    with open(os.path.join(latest_model_dir, 'metadata.json'), 'r') as f:
        metadata = json.load(f)
    print(f"Model metrics: {metadata['metrics']}")
    print(f"Hyperparameters: {metadata['hyperparameters']}")
else:
    print("No model found! Run notebook 4 first.")

Found model: /home/adnoman/projects/aml_gan/AMLend2end/models/node_embeddings_e2366de4
Model metrics: {'accuracy': 0.8891156462585034}
Hyperparameters: {'walk_number': 2, 'walk_length': 2, 'emb_size': 32}


In [4]:
# Load node embeddings from notebook 4
embeddings_path = os.path.join(TRAINING_DATA_PATH, "node_embeddings.csv")
node_embeddings_df = pd.read_csv(embeddings_path)

print(f"Loaded embeddings shape: {node_embeddings_df.shape}")
node_embeddings_df.head()

Loaded embeddings shape: (7347, 33)


,node_id,emb_0,emb_1,emb_2,emb_3,emb_4,emb_5,emb_6,emb_7,emb_8,...,emb_22,emb_23,emb_24,emb_25,emb_26,emb_27,emb_28,emb_29,emb_30,emb_31
0,3aa9646b,-0.008853,0.004043,-0.003818,0.007575,0.023354,-0.027285,-0.005630,0.025699,0.025976,...,-0.008803,0.029562,-0.011895,0.020304,-0.001925,0.007217,-0.029996,0.016181,0.015940,0.008504
1,1e46e726,0.022349,-0.001167,0.030925,-0.012870,0.030629,0.014389,0.023963,0.009350,-0.012923,...,0.020407,-0.005137,0.017466,0.022701,-0.026283,0.017401,0.015713,-0.000234,-0.007488,0.008504
2,49203bc3,-0.005501,-0.009205,-0.027755,-0.014818,-0.012859,-0.024729,0.023249,0.016355,0.025465,...,0.017831,-0.015878,-0.013799,-0.000237,0.019603,0.026605,-0.015244,0.002540,0.007230,0.006761
3,a74d1101,0.011437,-0.000023,0.007053,-0.003801,0.022929,0.025165,-0.000650,-0.014209,-0.006374,...,0.007219,-0.027126,0.013434,0.008401,-0.026911,-0.028524,0.011721,0.026398,-0.003113,-0.029955
4,616d4505,-0.017530,-0.002801,-0.000988,-0.024782,-0.026430,-0.030293,0.005277,0.004350,0.028420,...,0.018822,0.026174,-0.028986,-0.003035,-0.015092,0.024190,0.002712,0.019147,0.031202,-0.002625


## Define model and load wights 

In [5]:
# Get embedding columns
emb_cols = [c for c in node_embeddings_df.columns if c.startswith('emb_')]
print(f"Embedding dimensions: {len(emb_cols)}")

# Preview embeddings
node_embeddings_df[['node_id'] + emb_cols[:5]].head()

Embedding dimensions: 32


,node_id,emb_0,emb_1,emb_2,emb_3,emb_4
0,3aa9646b,-0.008853,0.004043,-0.003818,0.007575,0.023354
1,1e46e726,0.022349,-0.001167,0.030925,-0.012870,0.030629
2,49203bc3,-0.005501,-0.009205,-0.027755,-0.014818,-0.012859
3,a74d1101,0.011437,-0.000023,0.007053,-0.003801,0.022929
4,616d4505,-0.017530,-0.002801,-0.000988,-0.024782,-0.026430


## connect hsfs library and get fs handle

In [6]:
# Load alert nodes to join with embeddings
alert_nodes_df = pd.read_csv(os.path.join(TRAINING_DATA_PATH, "alert_nodes_td.csv"))
print(f"Alert nodes: {len(alert_nodes_df)}")
print(f"SAR nodes: {alert_nodes_df['is_sar'].sum()}")

Alert nodes: 7347
SAR nodes: 816


### Get node and edge traininhg dataset objects 

In [7]:
# Create embedding array column (for compatibility with original format)
node_embeddings_df['embedding'] = node_embeddings_df[emb_cols].values.tolist()

# Rename node_id to id for consistency
node_embeddings_df = node_embeddings_df.rename(columns={'node_id': 'id'})

# Select final columns
node_embeddings_final = node_embeddings_df[['id', 'embedding']].copy()
print(f"Final embeddings shape: {node_embeddings_final.shape}")
node_embeddings_final.head()

Final embeddings shape: (7347, 2)


,id,embedding
0,3aa9646b,"[-0.008852957, 0.004042693, -0.0038181087, 0.0..."
1,1e46e726,"[0.022348989, -0.0011668114, 0.030925123, -0.0..."
2,49203bc3,"[-0.005501451, -0.009204879, -0.027754912, -0...."
3,a74d1101,"[0.01143696, -2.2989218e-05, 0.0070530414, -0...."
4,616d4505,"[-0.017530298, -0.0028010544, -0.0009879462, -..."


### Read training datasets as pandas df 

In [8]:
# Join embeddings with alert nodes info
embeddings_with_labels = node_embeddings_df.merge(
    alert_nodes_df[['id', 'is_sar']], 
    on='id', 
    how='left'
)
embeddings_with_labels['is_sar'] = embeddings_with_labels['is_sar'].fillna(0).astype(int)

print(f"Embeddings with labels: {embeddings_with_labels.shape}")
print(f"SAR nodes in embeddings: {embeddings_with_labels['is_sar'].sum()}")

Embeddings with labels: (7347, 35)
SAR nodes in embeddings: 816


### Read hyperparamenter for graph embeddings

In [9]:
# Preview the data
print("Sample of embeddings with SAR labels:")
embeddings_with_labels[['id', 'is_sar'] + emb_cols[:3]].head(10)

Sample of embeddings with SAR labels:


,id,is_sar,emb_0,emb_1,emb_2
0,3aa9646b,0,-0.008853,0.004043,-0.003818
1,1e46e726,0,0.022349,-0.001167,0.030925
2,49203bc3,0,-0.005501,-0.009205,-0.027755
3,a74d1101,1,0.011437,-0.000023,0.007053
4,616d4505,0,-0.017530,-0.002801,-0.000988
5,99af2455,1,0.011358,0.009435,-0.023467
6,39be1ea2,0,0.009941,0.001355,-0.030964
7,e7ec7bdb,1,0.014966,-0.022827,0.012702
8,e2e0d938,0,0.012979,-0.014054,-0.016346
9,afc399a9,0,-0.025564,-0.007712,0.019848


### Construct stellargraph Graph object

In [10]:
# Statistics
print("Embedding statistics:")
print(f"  Total nodes: {len(embeddings_with_labels)}")
print(f"  SAR nodes (is_sar=1): {embeddings_with_labels['is_sar'].sum()}")
print(f"  Non-SAR nodes (is_sar=0): {(embeddings_with_labels['is_sar']==0).sum()}")
print(f"  Embedding dimensions: {len(emb_cols)}")

Embedding statistics:
  Total nodes: 7347
  SAR nodes (is_sar=1): 816
  Non-SAR nodes (is_sar=0): 6531
  Embedding dimensions: 32


### infer node embeddings

In [11]:
# Prepare final feature group data
# Keep id, all embedding columns, and is_sar
final_cols = ['id'] + emb_cols + ['is_sar']
node_embeddings_fg_df = embeddings_with_labels[final_cols].copy()

print(f"Feature group shape: {node_embeddings_fg_df.shape}")
node_embeddings_fg_df.head()

Feature group shape: (7347, 34)


,id,emb_0,emb_1,emb_2,emb_3,emb_4,emb_5,emb_6,emb_7,emb_8,...,emb_23,emb_24,emb_25,emb_26,emb_27,emb_28,emb_29,emb_30,emb_31,is_sar
0,3aa9646b,-0.008853,0.004043,-0.003818,0.007575,0.023354,-0.027285,-0.005630,0.025699,0.025976,...,0.029562,-0.011895,0.020304,-0.001925,0.007217,-0.029996,0.016181,0.015940,0.008504,0
1,1e46e726,0.022349,-0.001167,0.030925,-0.012870,0.030629,0.014389,0.023963,0.009350,-0.012923,...,-0.005137,0.017466,0.022701,-0.026283,0.017401,0.015713,-0.000234,-0.007488,0.008504,0
2,49203bc3,-0.005501,-0.009205,-0.027755,-0.014818,-0.012859,-0.024729,0.023249,0.016355,0.025465,...,-0.015878,-0.013799,-0.000237,0.019603,0.026605,-0.015244,0.002540,0.007230,0.006761,0
3,a74d1101,0.011437,-0.000023,0.007053,-0.003801,0.022929,0.025165,-0.000650,-0.014209,-0.006374,...,-0.027126,0.013434,0.008401,-0.026911,-0.028524,0.011721,0.026398,-0.003113,-0.029955,1
4,616d4505,-0.017530,-0.002801,-0.000988,-0.024782,-0.026430,-0.030293,0.005277,0.004350,0.028420,...,0.026174,-0.028986,-0.003035,-0.015092,0.024190,0.002712,0.019147,0.031202,-0.002625,0


In [12]:
# Dummy cell - removed pyspark code

In [13]:
# Dummy cell - removed pyspark code

In [14]:
# Dummy cell - removed pyspark code

In [15]:
# Dummy cell - removed pyspark code

In [16]:
# Dummy cell - removed pyspark code

## Create embeddings feature group

In [17]:
# Save node embeddings feature group locally (replaces hsfs)
fg_path = os.path.join(OUTPUT_PATH, "node_embeddings_fg.parquet")
node_embeddings_fg_df.to_parquet(fg_path, index=False)
print(f"Saved node embeddings feature group to: {fg_path}")

# Also save as CSV for easier inspection
csv_path = os.path.join(OUTPUT_PATH, "node_embeddings_fg.csv")
node_embeddings_fg_df.to_csv(csv_path, index=False)
print(f"Saved CSV version to: {csv_path}")

Saved node embeddings feature group to: /home/adnoman/projects/aml_gan/AMLend2end/output/node_embeddings_fg.parquet


Saved CSV version to: /home/adnoman/projects/aml_gan/AMLend2end/output/node_embeddings_fg.csv


In [18]:
# Summary
print("=" * 50)
print("Node Embeddings Feature Group Created")
print("=" * 50)
print(f"Total nodes: {len(node_embeddings_fg_df)}")
print(f"Embedding dimensions: {len(emb_cols)}")
print(f"SAR nodes: {node_embeddings_fg_df['is_sar'].sum()}")
print(f"Non-SAR nodes: {(node_embeddings_fg_df['is_sar']==0).sum()}")
print(f"\nSaved to: {fg_path}")
print("=" * 50)

Node Embeddings Feature Group Created
Total nodes: 7347
Embedding dimensions: 32
SAR nodes: 816
Non-SAR nodes: 6531

Saved to: /home/adnoman/projects/aml_gan/AMLend2end/output/node_embeddings_fg.parquet


## Feature group provenance
![Feature group provenance](./images/provenance_fg.png)

In [19]:
# Done!